In [12]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from dotenv import load_dotenv
import os
import time

In [2]:
# Leer el archivo .env para obtener las rutas de los documentos y la base de datos
load_dotenv()
DOCS_DIR = os.getenv("DOCS_DIR")
DB_DIR = os.getenv("DB_DIR")

In [3]:
print(f"Documentos en: {DOCS_DIR}")
print(f"Base de datos en: {DB_DIR}")

Documentos en: .\\docs
Base de datos en: .\\db


## Parte 1: Procesamiento de Documentos

In [4]:
# Cargar los documentos PDF
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

In [16]:
# Crear chunks de los documentos con repetición de 150 caracteres para evitar perder contexto
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
chunks = text_splitter.split_documents(documents)

In [18]:
# 3. Embeddings y ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=DB_DIR)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10144.34it/s]


In [19]:
# Prueba al vector store
query = "¿Cuándo inicia el proceso de matrícula para el ciclo 2026-1?"
print("Resultados modelo 1")
start_time_1 = time.time()
docs_1 = vector_store.similarity_search(query, k=3)
end_time_1 = time.time()
for i, doc in enumerate(docs_1):
    print(f"Chunk {i}:", doc.page_content+"\n")
print(f"Time taken (model 1): {end_time_1 - start_time_1:.4f} seconds")

Resultados modelo 1
Chunk 0: INICIO DE MATRÍCULA – CICLO 2026-1 
 
 
El martes 10 de marzo se inicia la Matrícula del Ci clo 202 6-1, el proceso de 
inscripción culmina el jueves 12 de marzo (9:00p.m.), los estudiantes deberán definir 
certeramente en qué cursos prefiere matricularse. Tener presente que la publicación 
de la lista de cursos permitidos considerando las notas del Ciclo de Verano 2026 será 
el 10 de marzo. 
En e l semestre 202 6-1, los cursos presentan actividades presenciales  y

Chunk 1: INICIO DE MATRÍCULA – CICLO 2026-1 
 
 
El martes 10 de marzo se inicia la Matrícula del Ci clo 202 6-1, el proceso de 
inscripción culmina el jueves 12 de marzo (9:00p.m.), los estudiantes deberán definir 
certeramente en qué cursos prefiere matricularse. Tener presente que la publicación 
de la lista de cursos permitidos considerando las notas del Ciclo de Verano 2026 será 
el 10 de marzo. 
En e l semestre 202 6-1, los cursos presentan actividades presenciales  y 
eventualmente algún 

In [20]:
# El motor de búsqueda (en la base de datos)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [21]:
# 4. El LLM y el Prompt
llm = ChatOllama(model="qwen3:0.6b", temperature=0.1)

In [9]:
# 5. Crear el Prompt y la Cadena (Chain)
system_prompt = (
    "Eres un asistente virtual experto en el proceso de matrícula de la PUCP.\n"
    "Responde la pregunta del estudiante usando solo el contexto provisto.\n"
    "Si no encuentras la respuesta en el contexto, di amablemente que no tienes esa información.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# --- PRUEBA DEL CHATBOT ---
# Prueba con una pregunta que sepas que está dentro de tus PDFs de "AvisoInicioDeMatricula"
pregunta = input()
print(f"\nPregunta del Estudiante: {pregunta}")

respuesta = rag_chain.invoke({"input": pregunta})
print(f"\nRespuesta del Chatbot:\n{respuesta['answer']}")


Pregunta del Estudiante: como es la calificacion en el curso de Derecho para ingenieros?

Respuesta del Chatbot:
No tengo información sobre la calificación en el curso de Derecho para ingenieros. El contexto proporcionado incluye cursos en informática, programación y ciencias de software, pero no se menciona nada relacionado con derecho o ingeniería.
